# Task 3.1


In [ ]:
import pandas as pd

df = pd.read_csv("/content/appointments_clean.csv")

df.shape

(990, 16)

In [ ]:
df.head()

,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
3,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
4,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1


In [ ]:
X = df[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type"
    ]
].copy()


X["reminder_sent"] = X["reminder_sent"].map({
    "Yes": 1,
    "No": 0
})

X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"]
)


y = df["no_show"]


print("X shape:", X.shape)
print("y shape:", y.shape)


print("\nFeature columns:")
print(X.columns.tolist())

X shape: (990, 11)
y shape: (990,)

Feature columns:
['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (792, 11)
Testing features: (198, 11)
Training target: (792,)
Testing target: (198,)


In [ ]:
from sklearn.metrics import accuracy_score

baseline_predictions = [0] * len(y_test)

baseline_accuracy = accuracy_score(y_test, baseline_predictions)

test_no_shows = (y_test == 1).sum()

print("Test set size:", len(y_test))
print("Actual no-shows in test set:", test_no_shows)
print("Baseline accuracy:", round(baseline_accuracy, 4))

Test set size: 198
Actual no-shows in test set: 49
Baseline accuracy: 0.7525


# Task 3.2

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)

X_test_s = scaler.transform(X_test)

print("Scaled training shape:", X_train_s.shape)
print("Scaled testing shape:", X_test_s.shape)

Scaled training shape: (792, 11)
Scaled testing shape: (198, 11)


In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train_s, y_train)

y_pred = model.predict(X_test_s)

print("Predictions:", y_pred[:10])

print("Actual:", y_test.iloc[:10].values)

Predictions: [0 0 1 0 0 0 0 0 0 0]
Actual: [0 0 0 0 0 1 0 1 0 0]


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

accuracy = accuracy_score(y_test, y_pred)

cm = confusion_matrix(y_test, y_pred)

precision = precision_score(y_test, y_pred)

recall = recall_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred)

y_prob = model.predict_proba(X_test_s)[:, 1]

roc_auc = roc_auc_score(y_test, y_prob)

print("Accuracy:", round(accuracy, 3))
print("\nConfusion Matrix:\n", cm)
print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))
print("F1-score:", round(f1, 3))
print("ROC-AUC:", round(roc_auc, 3))

Accuracy: 0.788

Confusion Matrix:
 [[139  10]
 [ 32  17]]
Precision: 0.63
Recall: 0.347
F1-score: 0.447
ROC-AUC: 0.789


In [ ]:
import numpy as np

feature_names = X.columns

coefficients = model.coef_[0]

for feature, coef in zip(feature_names, coefficients):
    print(feature, ":", round(coef, 3))

reminder_index = list(feature_names).index("reminder_sent")

reminder_coef = coefficients[reminder_index]

scaled_odds_ratio = np.exp(reminder_coef)

actual_odds_ratio = np.exp(
    reminder_coef / scaler.scale_[reminder_index]
)

print("\nReminder coefficient:", round(reminder_coef, 3))
print("Odds ratio (scaled):", round(scaled_odds_ratio, 3))
print("Odds ratio (No to Yes):", round(actual_odds_ratio, 3))

age : -0.254
distance_km : 0.253
fee_inr : 0.453
lead_days : 0.614
prior_no_shows : 0.65
reminder_sent : -0.581
booking_channel_App : -0.047
booking_channel_Phone : 0.216
booking_channel_Walk-In : -0.186
visit_type_Consultation : 0.016
visit_type_Follow-up : -0.016

Reminder coefficient: -0.581
Odds ratio (scaled): 0.56
Odds ratio (No to Yes): 0.309


The reminder_sent feature has the negative coefficient at -0.581. The exponential of this coefficient is 0.560, which shows how the odds change for a one-standard-deviation increase in the scaled feature. When we divide the coefficient by its deviation, which is about 0.49, we get an actual odds ratio of 0.309. This means that, in this model, appointments with reminders are linked to 69% lower odds of a no-show. This relationship is an association. It does not prove that sending reminders cause no-shows, because the data come from an observational study.

# Task 3.3


In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_full = DecisionTreeClassifier(random_state=42)

dt_full.fit(X_train, y_train)

train_pred_full = dt_full.predict(X_train)

test_pred_full = dt_full.predict(X_test)

train_acc_full = accuracy_score(y_train, train_pred_full)

test_acc_full = accuracy_score(y_test, test_pred_full)

print("Full-depth Tree")
print("Training accuracy:", round(train_acc_full, 3))
print("Test accuracy:", round(test_acc_full, 3))
print("Tree depth:", dt_full.get_depth())

Full-depth Tree
Training accuracy: 1.0
Test accuracy: 0.672
Tree depth: 15


In [ ]:
dt_depth3 = DecisionTreeClassifier(    max_depth=3,random_state=42)

dt_depth3.fit(X_train, y_train)

train_pred_depth3 = dt_depth3.predict(X_train)

test_pred_depth3 = dt_depth3.predict(X_test)

train_acc_depth3 = accuracy_score(y_train, train_pred_depth3)

test_acc_depth3 = accuracy_score(y_test, test_pred_depth3)

print("Depth-3 Tree")
print("Training accuracy:", round(train_acc_depth3, 3))
print("Test accuracy:", round(test_acc_depth3, 3))
print("Tree depth:", dt_depth3.get_depth())

Depth-3 Tree
Training accuracy: 0.822
Test accuracy: 0.778
Tree depth: 3


The depth Decision Tree got 100% accuracy on the training data but only about 67% on the test data. That is a sign of overfitting. It even came in below the 75.25% baseline accuracy. The depth-3 Decision Tree got around 82% training accuracy and 78% test accuracy. I think that shows generalisation, because keeping the tree simple cut down on the overfitting.

# Task 3.4


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100,random_state=42)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

rf_train_acc = accuracy_score(y_train, rf.predict(X_train))

rf_test_acc = accuracy_score(y_test, rf_pred)

rf_recall = recall_score(y_test, rf_pred)

rf_prob = rf.predict_proba(X_test)[:, 1]

rf_auc = roc_auc_score(y_test, rf_prob)

print("Training accuracy:", round(rf_train_acc, 3))
print("Test accuracy:", round(rf_test_acc, 3))
print("Recall:", round(rf_recall, 3))
print("ROC-AUC:", round(rf_auc, 3))

Training accuracy: 1.0
Test accuracy: 0.747
Recall: 0.306
ROC-AUC: 0.706


In [ ]:
importances = rf.feature_importances_

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importances
})

feature_importance = feature_importance.sort_values(  by="Importance",ascending=False)

print(round(feature_importance.head(3),2))

       Feature  Importance
3    lead_days        0.22
0          age        0.20
1  distance_km        0.19


In [ ]:
# Create a comparison table for the three models
model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree (Depth 3)",
        "Random Forest"
    ],

    "Accuracy": [
        accuracy,
        test_acc_depth3,
        rf_test_acc
    ],

    "Recall": [
        recall,
        recall_score(y_test, test_pred_depth3),
        rf_recall
    ],

    "ROC-AUC": [
        roc_auc,
        roc_auc_score(
            y_test,
            dt_depth3.predict_proba(X_test)[:, 1]
        ),
        rf_auc
    ]
})

print(model_comparison.round(3))

                     Model  Accuracy  Recall  ROC-AUC
0      Logistic Regression     0.788   0.347    0.789
1  Decision Tree (Depth 3)     0.778   0.224    0.700
2            Random Forest     0.747   0.306    0.706


# Task 3.5

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 8]
}

rf_grid = GridSearchCV(
    rf_model,
    param_grid,
    cv=5,
    scoring="roc_auc"
)

rf_grid.fit(X_train, y_train)

print("Best Parameters:", rf_grid.best_params_)
print("Best CV ROC-AUC:", rf_grid.best_score_)

Best Parameters: {'max_depth': 3, 'n_estimators': 200}
Best CV ROC-AUC: 0.7673242251739405


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

lr_auc = cross_val_score(
    lr_pipe, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

dt_auc = cross_val_score(
    dt_depth3, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

rf_auc = cross_val_score(
    rf, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

tuned_rf_auc = cross_val_score(
    rf_grid.best_estimator_, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

# Model comparison table

cv_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree (Depth 3)",
        "Random Forest",
        "Tuned Random Forest"
    ],
    "CV ROC-AUC": [
        lr_auc,
        dt_auc,
        rf_auc,
        tuned_rf_auc
    ]
})

print(cv_comparison.round(2))

                     Model  CV ROC-AUC
0      Logistic Regression        0.77
1  Decision Tree (Depth 3)        0.68
2            Random Forest        0.75
3      Tuned Random Forest        0.77


Logistic Regression got the cross-validation ROC-AUC of 0.77. It performed better than the Decision Tree, Random Forest and Tuned Random Forest. Because of this, Logistic Regression was chosen as the candidate model. It will now be used for threshold tuning and further evaluation.

In [ ]:
from sklearn.model_selection import cross_val_predict

oof = cross_val_predict(
    lr_pipe,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

print("Total probabilities:", len(oof))
print("First 10 probabilities:", oof[:10])

Total probabilities: 792
First 10 probabilities: [0.44277273 0.11344644 0.34663886 0.12072607 0.17356387 0.24988488
 0.12508639 0.4103534  0.16669732 0.08633667]


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

pred_50 = (oof >= 0.50).astype(int)

precision_50 = precision_score(y_train, pred_50)
recall_50 = recall_score(y_train, pred_50)
f1_50 = f1_score(y_train, pred_50)

flagged_50 = pred_50.sum()

print("Threshold: 0.50")
print("Precision:", round(precision_50, 3))
print("Recall:", round(recall_50, 3))
print("F1-score:", round(f1_50, 3))
print("Flagged appointments:", flagged_50)

Threshold: 0.50
Precision: 0.575
Recall: 0.249
F1-score: 0.347
Flagged appointments: 73


In [ ]:
pred_35 = (oof >= 0.35).astype(int)

precision_35 = precision_score(y_train, pred_35)
recall_35 = recall_score(y_train, pred_35)
f1_35 = f1_score(y_train, pred_35)

flagged_35 = pred_35.sum()

print("Threshold: 0.35")
print("Precision:", round(precision_35, 3))
print("Recall:", round(recall_35, 3))
print("F1-score:", round(f1_35, 3))
print("Flagged appointments:", flagged_35)

Threshold: 0.35
Precision: 0.513
Recall: 0.467
F1-score: 0.489
Flagged appointments: 154


A threshold of 0.35 was chosen because it gave a F1-score compared to the 0.50 threshold. Even though this lower threshold means more appointments are flagged for follow-up, it boosts recall. This helps the clinic catch potential no-shows, which is important for planning and reducing missed appointments.

In [ ]:
lr_pipe.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model', LogisticRegression(max_iter=1000))])

In [ ]:
probs = lr_pipe.predict_proba(X_test)[:, 1]

print("Total test probabilities:", len(probs))
print("First 10 probabilities:", probs[:10])

Total test probabilities: 198
First 10 probabilities: [0.02156542 0.15091659 0.51869182 0.19130033 0.16771795 0.31997718
 0.33844789 0.36703784 0.10499898 0.17326066]


In [ ]:
pred_test = (probs >= 0.35).astype(int)

print("Total predictions:", len(pred_test))
print("Flagged appointments:", pred_test.sum())

Total predictions: 198
Flagged appointments: 49


In [ ]:
precision_test = precision_score(y_test, pred_test)

recall_test = recall_score(y_test, pred_test)

print("Precision:", round(precision_test, 3))
print("Recall:", round(recall_test, 3))

Precision: 0.571
Recall: 0.571


In [ ]:
roc_auc_test = roc_auc_score(y_test, probs)

print("Test ROC-AUC:", round(roc_auc_test, 3))

Test ROC-AUC: 0.789


Our model flagged 49 out of 198 appointments for follow-up calls.

Approximately 57% of flagged appointments were actual no-shows, and the model identified approximately 57% of all actual no-shows.

The ROC-AUC of 0.79 indicates good ability to distinguish between the two appointment outcomes.

In [ ]:
print("X_test first 10 indices:")
print(X_test.index[:10].tolist())

print("\nX_test shape:")
print(X_test.shape)

X_test first 10 indices:
[213, 331, 501, 309, 88, 535, 280, 107, 59, 506]

X_test shape:
(198, 11)


In [ ]:
test_ids = df.loc[X_test.index, "appointment_id"]

print(test_ids.head(10))
print("\nTotal appointment IDs:", len(test_ids))

213    A0217
331    A0336
501    A0508
309    A0314
88     A0089
535    A0543
280    A0284
107    A0109
59     A0060
506    A0513
Name: appointment_id, dtype: object

Total appointment IDs: 198


In [ ]:
test_risk_scores = pd.DataFrame({
    "appointment_id": test_ids,
    "no_show": y_test,
    "risk_prob": probs,
    "flag": pred_test
})

print(test_risk_scores.head(10))
print("\nDataset shape:", test_risk_scores.shape)

    appointment_id  no_show  risk_prob  flag
213          A0217        0   0.021565     0
331          A0336        0   0.150917     0
501          A0508        0   0.518692     1
309          A0314        0   0.191300     0
88           A0089        0   0.167718     0
535          A0543        1   0.319977     0
280          A0284        0   0.338448     0
107          A0109        1   0.367038     1
59           A0060        0   0.104999     0
506          A0513        0   0.173261     0

Dataset shape: (198, 4)


In [ ]:
print("Total rows:", len(test_risk_scores))
print("Total columns:", len(test_risk_scores.columns))
print("Column names:", test_risk_scores.columns.tolist())
print("Flagged appointments:", test_risk_scores["flag"].sum())

# Check for missing values
print("\nMissing values:")
print(test_risk_scores.isnull().sum())

Total rows: 198
Total columns: 4
Column names: ['appointment_id', 'no_show', 'risk_prob', 'flag']
Flagged appointments: 49

Missing values:
appointment_id    0
no_show           0
risk_prob         0
flag              0
dtype: int64


In [ ]:
test_risk_scores.to_csv("test_risk_scores.csv", index=False)